In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

def highlight_max(row):
    is_max = row == row.max()
    return ['font-weight: bold' if v else '' for v in is_max]

def drop_nan_group(group):
    if group.isnull().any():
        return None  # Returning None will drop the group
    else:
        return group

def compte_les_points(df):
    results = {}
    total = 0
    for idx, row in df.reset_index()['perf'].iterrows():
        is_max = row == row.max()
        for key in is_max.reset_index()['package'].values:
            if key not in results:
                results[key] = 0
                
            results[key] += int(is_max[key])
        total += 1
            
    for key in results:
        print(f"{key} : {results[key]} / {total}")

# Clean

In [ ]:

bench = pd.read_csv("/Users/rudy/Documents/CHU/iias/baiddy_group/automl/src/perf_logger/tests_data/bench_v2.log")
print(bench.shape)
bench['perf'] = bench.apply(lambda row: row['balanced_accuracy'] if pd.notnull(row['balanced_accuracy']) else row['r2_score'], axis=1)
bench = bench[(bench['max_duration'] - bench['compute_time']) >= -30] # remove line with too long computing time
count_fold = bench.groupby(['package', 'max_duration', 'dataset']).count().reset_index()[['package', 'max_duration', 'dataset', 'fold']]

joined = bench.merge(count_fold, on=['package', 'max_duration', 'dataset'], suffixes=('', '_count'))
bench = joined[joined['fold_count'] == 5]

bench.head()

In [ ]:
bench['perf'] = bench.apply(lambda row: row['balanced_accuracy'] if pd.notnull(row['balanced_accuracy']) else row['r2_score'], axis=1)

# Analyse

# With folds

In [ ]:
merged_df = bench[['package', 'max_duration', 'dataset', 'perf', 'fold']]
grouped = merged_df.groupby(['package', 'max_duration', 'dataset', 'fold']).mean(numeric_only=True).reset_index()[['package', 'perf', 'max_duration', 'dataset', 'fold']].dropna()

# grouped
pivot_df = grouped.pivot(index=['dataset', 'max_duration', 'fold'], columns='package')#.dropna()
pivot_df.style.apply(highlight_max, axis=1)

# Mean of folds

In [ ]:
merged_df = bench[['package', 'max_duration', 'dataset', 'perf', 'compute_time']]
grouped = merged_df.groupby(['package', 'max_duration', 'dataset']).mean(numeric_only=True).reset_index()[['package', 'perf', 'max_duration', 'dataset']].dropna()

# grouped
pivot_df = grouped.pivot(index=['dataset', 'max_duration'], columns='package')#.dropna()
pivot_df.style.apply(highlight_max, axis=1)

In [ ]:
# def count(row):
#     is_max = row == row.max()
#     return ['font-weight: bold' if v else '' for v in is_max]

# pivot_df = grouped.pivot(index=['dataset_name', 'duration'], columns='package')

# pivot_df.style.apply(highlight_max, axis=1)

compte_les_points(pivot_df)

In [ ]:
# Auto-Sklearn : 14 / 75
# AutoMed : 29 / 75
# NaiveAutoML : 38 / 75

In [ ]:
# PAR DATASET sans prendre ne compte le temps
without_duration = grouped.drop(columns=['max_duration'])
without_duration

pivot_df = without_duration.groupby(['package', 'dataset']).max(numeric_only=True).reset_index().pivot(index=['dataset'], columns='package')#.dropna()
pivot_df.style.apply(highlight_max, axis=1)

# Attention Naive_AutoML à probablement plus de temps ci-dessus

In [ ]:
compte_les_points(pivot_df)

In [ ]:


def line_plot(df, filter_tuple, hue="dataset",zero_line=False, y_lim=None):
    plt.figure()
    plot = sns.lineplot(data=df[df[filter_tuple[0]] == filter_tuple[1]], x="max_duration", y="perf", hue=hue) 
    sns.move_legend(plot, "upper left", bbox_to_anchor=(1, 1))
    plt.title(str(filter_tuple))
    if zero_line:
        plt.axhline(y=0.0, color='r', linestyle='--')
        
    if y_lim:
        plt.ylim(*y_lim)
    plt.show()
    
def bar_plot(df, dataset, show_fold=False):
    if not show_fold:
        df = df.groupby(['package', 'max_duration', 'dataset']).mean(numeric_only=True).reset_index()
    
    package_order = sorted(df['package'].unique())
    plot = sns.barplot(data=df[df['dataset'] == dataset], x='max_duration', y='perf', hue='package', hue_order=package_order)
    sns.move_legend(plot, "upper left", bbox_to_anchor=(1, 1))
    plt.title(str(dataset))
    plt.show()


merged_df = bench[['package', 'max_duration', 'dataset', 'perf']]
merged_df = merged_df.dropna() # Avoid penalize a package if no value


In [ ]:
# plt.figure(figsize=(10, 6))
for dataset in merged_df['dataset'].unique():
    bar_plot(bench, dataset)

In [ ]:
# line_plot(merged_df, ("package", "AutoMed_void"))

In [ ]:
# line_plot(merged_df, ("package", "NaiveAutoML"))

In [ ]:
# line_plot(merged_df, ("package", "Auto-Sklearn"))

In [ ]:
# for dataset_name in merged_df['dataset_name'].unique():
#     line_plot(merged_df, ("dataset_name", dataset_name), hue="package")

In [ ]:
# Diff with automed ?
# Ensure df is a copy to avoid SettingWithCopyWarning
df = merged_df.copy()

# Separate 'automed' and other packages
automed_df = df[df['package'] == 'automed']
other_df = df[df['package'] != 'automed']

# Average 'automed' performance for each 'duration' and 'dataset_name' combination
automed_avg_perf = automed_df.groupby(['max_duration', 'dataset'])['perf'].mean()

# Merge this average performance back into the other_df based on 'max_duration' and 'dataset'
other_df = other_df.merge(automed_avg_perf, on=['max_duration', 'dataset'], suffixes=('', '_automed_avg'))

# Calculate the difference in performance
other_df['perf'] = other_df['perf'] - other_df['perf_automed_avg']

# Drop the now unnecessary 'perf_automed_avg' column
other_df.drop(columns=['perf_automed_avg'], inplace=True)

# Resulting DataFrame has 'automed' rows removed and 'perf' adjusted
df_result = other_df

In [ ]:
for dataset in merged_df['dataset'].unique():
    try:
        line_plot(df_result, ("dataset", dataset), hue="package", zero_line=True)#, y_lim=(-0.2, 0.2))
    except:
        print(f"error for {dataset}")

# Mean by package and duration

In [ ]:
pivot_df = bench.groupby(['package', 'dataset', 'max_duration']).mean(numeric_only=True).reset_index().groupby(['package', 'max_duration']).mean(numeric_only=True).reset_index()

mean_by_package = pivot_df[['package', 'max_duration', 'perf']]
mean_by_duration = mean_by_package.groupby(['max_duration']).mean(numeric_only=True)

joined = mean_by_package.merge(mean_by_duration, on=['max_duration'], suffixes=('', '_by_duration'))
joined['perf'] = joined['perf'] - joined['perf_by_duration']

plt.figure()
plot = sns.lineplot(data=joined, x="max_duration", y="perf", hue="package") 
sns.move_legend(plot, "upper left", bbox_to_anchor=(1, 1))
plt.axhline(y=0.0, color='r', linestyle='--')
plt.show()

# Rank approch

In [ ]:
with_rank = bench.copy().groupby(['package', 'max_duration', 'dataset']).mean(numeric_only=True).reset_index()[['max_duration', 'dataset', 'perf', 'package']]

# Remove dataset with nan
with_rank = with_rank.pivot(index=['dataset', 'max_duration'], columns='package').dropna().stack().reset_index()

# Sorting by perf within each group and then ranking
with_rank['rank'] = with_rank.sort_values('perf', ascending=False) \
               .groupby(['max_duration', 'dataset'])['perf'] \
               .rank(method='first', ascending=True)

# Optionally, sort the DataFrame by max_duration, dataset, and rank to see the ranking clearly
with_rank_sorted = with_rank.sort_values(by=['dataset', 'max_duration', 'rank'])

with_rank_sorted = with_rank_sorted[['max_duration', 'dataset', 'rank', 'package']].groupby(['package', 'max_duration']).mean(numeric_only=True)

plot = sns.barplot(data=with_rank_sorted, x='max_duration', y='rank', hue='package')
sns.move_legend(plot, "upper left", bbox_to_anchor=(1, 1))
plt.title('Mean rank by duration')
plt.show()